# ETL Pipeline: Certificate Data Integration (Factory to Gold)
### Project: Global HR Data Lakehouse

**Description**: This notebook implements the final ETL logic to consolidate employee license and certification data from heterogeneous systems (System A and System B). The integrated data is then persisted into the **Gold Layer** (Delta Lake) for analytical reporting.

**Key Engineering Steps**:
* **Schema Normalization**: Aligning disparate schemas from two legacy systems.
* **Data Consolidation**: Merging records using `UNION` logic with standardized ID prefixing.
* **Persistence**: Writing to ADLS Gen2 in Delta format and registering the global table.

## 1. Data Transformation (Python & Spark SQL)
We use Spark SQL to handle complex joins and union logic for schema alignment.

In [ ]:
# Databricks Cell: Python
# Integration Query for Consolidated License Information
df = spark.sql("""
-- Logic for System B (Refining Bronze Layer data)
SELECT
    LICENSE.MOD_USER_ID  AS MOD_PRSN,
    LICENSE.MOD_DATE AS MOD_DTT,
    LICENSE.TZ_CD AS TZ_CD,
    LICENSE.TZ_DATE AS TZ_DTT,
    CAST(LICENSE.EMP_ID AS string ) AS EMP_ID,
    LICENSE.PHM_LICENSE_ID AS EMP_LCNS_ID,
    LICENSE.LICENSE_COMPANY_NM AS ISSUE_ORG,
    LICENSE.LICENSE_TYPE_CD AS LCNS_DIV_CD,
    COMM_LICENSE_TY.CD_NM AS LCNS_DIV_NM,
    COMM_LICENSE.CD_NM AS LCNS_NM,
    LICENSE.NOTE AS NOTE,
    LICENSE.BONUS_TYPE AS BONUS_PAY_DIV_CD,
    LICENSE.REG_YN AS REG_YN,
    LICENSE.LICENSE_NO AS LCNS_NO,
    to_timestamp(LICENSE.END_YMD , 'yyyyMMdd') AS VALID_DT,
    to_timestamp(LICENSE.STA_YMD , 'yyyyMMdd') AS GET_DT,
    LICENSE.PERSON_ID  AS PRSNL_ID,
    LICENSE.LICENSE_CD AS LCNS_CD
FROM brz.system_b.PHM_LICENSE LICENSE
LEFT JOIN brz.system_b.VE_FRM_CODE COMM_LICENSE
  ON COMM_LICENSE.CD_KIND ='PHM_LICENSE_CD' AND LICENSE.LICENSE_CD = COMM_LICENSE.CD
LEFT JOIN brz.system_b.VE_FRM_CODE COMM_LICENSE_TY
  ON COMM_LICENSE_TY.CD_KIND ='PHM_LICENSE_TYPE_CD' AND LICENSE.LICENSE_TYPE_CD = COMM_LICENSE_TY.CD

UNION ALL

-- Logic for System A (Refining Silver Layer data with Company-based ID Prefixing)
SELECT
    NULL as MOD_PRSN,
    l.UPDATE_DTS as MOD_DTT,
    NULL as TZ_CD,
    NULL as TZ_DTT,
    CASE
        WHEN l.COMPANY_CD = '3000' THEN 'a' || l.EMP_NO -- Subsidiary A
        WHEN l.COMPANY_CD = '2000' THEN 'q' || l.EMP_NO -- Subsidiary B
        WHEN l.COMPANY_CD = '1000' THEN 'f' || l.EMP_NO -- Subsidiary C
        ELSE l.EMP_NO
    END AS EMP_ID,
    NULL as EMP_LCNS_ID,
    l.ISSUE_ISTN_NM as ISSUE_ORG,
    NULL as LCNS_DIV_CD,
    NULL as LCNS_DIV_NM,
    l.QUA_CARD_NM as LCNS_NM,
    l.RMK_DC as NOTE,
    NULL as BONUS_PAY_DIV_CD,
    NULL as REG_YN,
    NULL as LCNS_NO,
    to_timestamp(l.VLID_DT , 'yyyyMMdd') as VALID_DT,
    to_timestamp(l.ACQS_DT , 'yyyyMMdd') as GET_DT,
    NULL as PRSNL_ID,
    NULL as LCNS_CD
FROM slv.system_a.hr_license_mst l
WHERE l.COMPANY_CD in ('1000', '2000', '3000')
""")

# Writing transformed data to Gold Layer (Delta format)
df.write.mode("overwrite").save("abfss://gld@storage_account.dfs.core.windows.net/f_hr_license")

## 2. Table Registration and Validation
Registering the persisted data as a Spark table for downstream analytics and verifying record counts.

In [ ]:
-- Databricks Cell: SQL
-- Create External Delta Table
USE gld.default;

CREATE TABLE IF NOT EXISTS f_hr_license
USING DELTA
LOCATION 'abfss://gld@storage_account.dfs.core.windows.net/f_hr_license';

-- Final Count Validation
SELECT COUNT(*) AS total_records FROM gld.default.f_hr_license;